In [ ]:
#Models included in this file are : UnlimitedOCR and OlmOCR 2

In [3]:
import os
import pandas as pd

DATASET_DIR = "/kaggle/input/datasets/manyabhatnagar/inputs1"

CSV_PATH = os.path.join(DATASET_DIR, "ClinOCR_Dataset.csv")
IMAGE_DIR = os.path.join(DATASET_DIR, "ClinOCR_images")

print("CSV exists:", os.path.exists(CSV_PATH))
print("Image directory exists:", os.path.exists(IMAGE_DIR))
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV exists: True
Image directory exists: True
Number of images: 192


In [4]:
DATASET_DIR = "/kaggle/input/datasets/manyabhatnagar/inputs1"
IMAGE_DIR = os.path.join(DATASET_DIR, "ClinOCR_images")
CSV_PATH = os.path.join(DATASET_DIR, "ClinOCR_Dataset.csv")

df = pd.read_csv(CSV_PATH)

df["image"] = df["image"].apply(
    lambda x: os.path.join(IMAGE_DIR, os.path.basename(x))
)

print("Rows:", len(df))
print("Images found:", df["image"].apply(os.path.exists).sum())
print("Images missing:", (~df["image"].apply(os.path.exists)).sum())

display(df[["doc_id", "image", "ground_truth"]].head())

Rows: 192
Images found: 192
Images missing: 0


,doc_id,image,ground_truth
0,normal_t1_s1,/kaggle/input/datasets/manyabhatnagar/inputs1/...,"Patient Name: Doe, Jane Patient ID: JG05141985..."
1,normal_t2_s1,/kaggle/input/datasets/manyabhatnagar/inputs1/...,Lakewood Hospital - Pathology Pathology Report...
2,normal_t3_s1,/kaggle/input/datasets/manyabhatnagar/inputs1/...,Riverside Hospital. 742 Evergreen Terrace Spri...
3,normal_t4_s1,/kaggle/input/datasets/manyabhatnagar/inputs1/...,GENERALHOSPITAL\nHOSPITAL\nDate of Visit: 06/2...
4,normal_t5_s1,/kaggle/input/datasets/manyabhatnagar/inputs1/...,"University Medical Center 456 Oak Avenue, Spri..."


In [ ]:
!pip install -q \
    torch==2.10.0 \
    torchvision==0.25.0 \
    transformers==4.57.1 \
    matplotlib==3.10.8 \
    einops==0.8.2 \
    addict==2.4.0 \
    easydict==1.13 \
    pymupdf==1.27.2.2 \
    psutil==7.2.2

import PIL
import os

print("Pillow version:", PIL.__version__)
print("Pillow location:", os.path.dirname(PIL.__file__))

import PIL._typing as t

print("_Ink exists:", hasattr(t, "_Ink"))

In [ ]:
!pip install --force-reinstall --no-cache-dir --no-deps Pillow==12.1.1

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "baidu/Unlimited-OCR"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)

model = model.eval().cuda()

print("Unlimited-OCR loaded.")
print("Model:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Dtype :", next(model.parameters()).dtype)

In [ ]:
test_image = df.iloc[0]["image"]

print("Image:", test_image)

result = model.infer(
    tokenizer,
    prompt="<image>document parsing.",
    image_file=test_image,
    output_path="/kaggle/working/unlimited_ocr_test",
    base_size=1024,
    image_size=640,
    crop_mode=True,
    max_length=32768,
    no_repeat_ngram_size=35,
    ngram_window=128,
    save_results=True,
)

print("\nRETURNED RESULT:")
print(result)

In [ ]:
def benchmark_unlimited_ocr(
    df,
    model,
    tokenizer,
    raw_output_dir="/kaggle/working/unlimited_ocr_raw",
    image_dir="/kaggle/input/datasets/manyabhatnagar/inputs1/ClinOCR_images",
    prompt="<image>document parsing.",
    base_size=1024,
    image_size=640,
    crop_mode=True,
    max_length=32768,
    no_repeat_ngram_size=35,
    ngram_window=128,
):
    import os
    import re
    import time
    import shutil
    import warnings
    import zipfile
    import io

    import torch
    import pandas as pd

    from contextlib import redirect_stdout
    from tqdm.auto import tqdm

    warnings.filterwarnings("ignore")

    # ==========================================================
    # CLEAN PREVIOUS RUN
    # ==========================================================

    if os.path.exists(raw_output_dir):
        shutil.rmtree(raw_output_dir)

    os.makedirs(raw_output_dir, exist_ok=True)

    csv_path = "/kaggle/working/unlimited_ocr_results.csv"
    zip_path = "/kaggle/working/unlimited_ocr_raw.zip"

    if os.path.exists(csv_path):
        os.remove(csv_path)

    if os.path.exists(zip_path):
        os.remove(zip_path)

    # ==========================================================
    # COPY DATAFRAME
    # ==========================================================

    results_df = df.copy()

    for col in [
        "prediction",
        "runtime",
        "raw_output",
        "model",
    ]:
        if col not in results_df.columns:
            results_df[col] = None

    total = len(results_df)

    print("=" * 70, flush=True)
    print("UNLIMITED-OCR BENCHMARK", flush=True)
    print("=" * 70, flush=True)
    print(f"Total documents : {total}", flush=True)
    print(f"Raw output dir  : {raw_output_dir}", flush=True)
    print("=" * 70, flush=True)

    # ==========================================================
    # PROGRESS BAR
    #
    # IMPORTANT:
    # We use tqdm and DO NOT redirect stderr.
    # Unlimited-OCR's tqdm output therefore remains visible.
    # ==========================================================

    pbar = tqdm(
        total=total,
        desc="Unlimited-OCR",
        unit="doc",
        dynamic_ncols=True,
    )

    successful = 0
    failed = 0

    # ==========================================================
    # BENCHMARK
    # ==========================================================

    for counter, idx in enumerate(
        results_df.index,
        start=1
    ):

        row = results_df.loc[idx]

        doc_id = str(row["doc_id"])

        filename = os.path.basename(
            str(row["image"])
        )

        image_path = os.path.join(
            image_dir,
            filename
        )

        pbar.set_postfix(
            completed=successful,
            failed=failed,
            current=doc_id
        )

        # ------------------------------------------------------
        # IMAGE CHECK
        # ------------------------------------------------------

        if not os.path.exists(image_path):

            results_df.at[
                idx,
                "prediction"
            ] = ""

            results_df.at[
                idx,
                "runtime"
            ] = None

            results_df.at[
                idx,
                "raw_output"
            ] = None

            results_df.at[
                idx,
                "model"
            ] = "Unlimited-OCR"

            failed += 1

            pbar.set_postfix(
                completed=successful,
                failed=failed,
                current=f"{doc_id} IMAGE NOT FOUND"
            )

            pbar.update(1)

            continue

        # ------------------------------------------------------
        # RAW OUTPUT FILE
        # ------------------------------------------------------

        raw_filename = f"{doc_id}.txt"

        raw_path = os.path.join(
            raw_output_dir,
            raw_filename
        )

        try:

            # ==================================================
            # INFERENCE
            #
            # CRITICAL:
            #
            # stdout -> captured
            # stderr -> NOT captured
            #
            # Therefore:
            #   OCR text is captured
            #   tqdm remains visible
            # ==================================================

            stdout_buffer = io.StringIO()

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            with redirect_stdout(stdout_buffer):

                with torch.inference_mode():

                    returned_result = model.infer(
                        tokenizer,
                        prompt=prompt,
                        image_file=image_path,
                        output_path=raw_output_dir,
                        base_size=base_size,
                        image_size=image_size,
                        crop_mode=crop_mode,
                        max_length=max_length,
                        no_repeat_ngram_size=no_repeat_ngram_size,
                        ngram_window=ngram_window,
                        save_results=False,
                    )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            runtime = (
                time.perf_counter()
                - start
            )

            # ==================================================
            # IMPORTANT:
            #
            # Unlimited-OCR returns None.
            #
            # DO NOT use returned_result.
            #
            # The actual OCR is in stdout_buffer.
            # ==================================================

            captured_stdout = (
                stdout_buffer.getvalue()
            )

            if not captured_stdout.strip():

                raise RuntimeError(
                    "Unlimited-OCR produced no stdout output."
                )

            # ==================================================
            # EXTRACT ACTUAL OCR
            #
            # Output looks like:
            #
            # <|det|>header [...]<|/det|>Patient...
            #
            # ...
            #
            # ===============save results:===============
            #
            # ==================================================

            save_marker = (
                "===============save results:==============="
            )

            if save_marker in captured_stdout:

                raw_text = captured_stdout.split(
                    save_marker,
                    1
                )[0]

            else:

                raw_text = captured_stdout

            # Remove anything before the first detection block
            first_det = raw_text.find("<|det|>")

            if first_det >= 0:

                raw_text = raw_text[
                    first_det:
                ]

            raw_text = raw_text.strip()

            if not raw_text:

                raise RuntimeError(
                    "Could not extract OCR text from "
                    "Unlimited-OCR stdout."
                )

            # ==================================================
            # SAVE EXACT OCR OUTPUT
            # ==================================================

            with open(
                raw_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(raw_text)

            # ==================================================
            # CLEAN PREDICTION
            #
            # Remove:
            #
            # <|det|>text [48, 60, 842, 75]<|/det|>
            #
            # while KEEPING:
            #
            # Patient Name: ...
            # ==================================================

            prediction = re.sub(
                r"<\|det\|>.*?\[.*?\]<\|/det\|>",
                "",
                raw_text,
                flags=re.DOTALL
            )

            # --------------------------------------------------
            # Remove any remaining det tags
            # --------------------------------------------------

            prediction = re.sub(
                r"<\|det\|>|<\|/det\|>",
                "",
                prediction
            )

            # --------------------------------------------------
            # Remove remaining special tokens
            # --------------------------------------------------

            prediction = re.sub(
                r"<\|.*?\|>",
                "",
                prediction,
                flags=re.DOTALL
            )

            # --------------------------------------------------
            # Clean whitespace
            # --------------------------------------------------

            prediction = re.sub(
                r"[ \t]+",
                " ",
                prediction
            )

            prediction = re.sub(
                r"\n{3,}",
                "\n\n",
                prediction
            )

            prediction = prediction.strip()

            # ==================================================
            # STORE RESULTS
            # ==================================================

            results_df.at[
                idx,
                "prediction"
            ] = prediction

            results_df.at[
                idx,
                "runtime"
            ] = runtime

            results_df.at[
                idx,
                "raw_output"
            ] = raw_filename

            results_df.at[
                idx,
                "model"
            ] = "Unlimited-OCR"

            successful += 1

            # ==================================================
            # UPDATE TQDM
            # ==================================================

            pbar.update(1)

            pbar.set_postfix(
                completed=successful,
                failed=failed,
                last=f"{runtime:.1f}s",
                chars=len(prediction)
            )

        # ======================================================
        # ERROR
        # ======================================================

        except Exception as e:

            failed += 1

            results_df.at[
                idx,
                "prediction"
            ] = ""

            results_df.at[
                idx,
                "runtime"
            ] = None

            results_df.at[
                idx,
                "raw_output"
            ] = None

            results_df.at[
                idx,
                "model"
            ] = "Unlimited-OCR"

            pbar.update(1)

            pbar.set_postfix(
                completed=successful,
                failed=failed,
                error=type(e).__name__
            )

            print(
                f"\nERROR on doc_id={doc_id}: "
                f"{type(e).__name__}: {e}",
                flush=True
            )

    # ==========================================================
    # CLOSE PROGRESS BAR
    # ==========================================================

    pbar.close()

    # ==========================================================
    # SAVE CSV
    # ==========================================================

    results_df.to_csv(
        csv_path,
        index=False
    )

    # ==========================================================
    # ZIP RAW OUTPUTS
    # ==========================================================

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as zipf:

        for filename in os.listdir(
            raw_output_dir
        ):

            filepath = os.path.join(
                raw_output_dir,
                filename
            )

            if os.path.isfile(filepath):

                zipf.write(
                    filepath,
                    arcname=filename
                )

    # ==========================================================
    # FINAL SUMMARY
    # ==========================================================

    print("\n" + "=" * 70, flush=True)
    print("UNLIMITED-OCR BENCHMARK COMPLETE", flush=True)
    print("=" * 70, flush=True)
    print(f"Total       : {total}", flush=True)
    print(f"Successful  : {successful}", flush=True)
    print(f"Failed      : {failed}", flush=True)
    print(f"CSV         : {csv_path}", flush=True)
    print(f"Raw outputs : {raw_output_dir}", flush=True)
    print(f"ZIP         : {zip_path}", flush=True)
    print("=" * 70, flush=True)

    return results_df

In [ ]:
unlimited_results = benchmark_unlimited_ocr(
    df,
    model,
    tokenizer
)

In [ ]:
!pip install -q "olmocr>=0.4.0"

In [ ]:
pip install compressed-tensors

In [6]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "allenai/olmOCR-2-7B-1025-FP8"

print("=" * 70)
print("Loading olmOCR-2 FP8")
print("=" * 70)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
)

model.eval()

print("\nModel loaded!")
print("Device map:")
print(model.hf_device_map)

print("\nGPU memory:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated"
    )

Loading olmOCR-2 FP8


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 680.02it/s]


Loading weights:   0%|          | 0/925 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]


Model loaded!
Device map:
{'model.visual': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 1, 'model.language_model.layers.8': 1, 'model.language_model.layers.9': 1, 'model.language_model.layers.10': 1, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_model.layers.24': 1

In [ ]:
import os
import torch
from PIL import Image
from olmocr.prompts import build_no_anchoring_v4_yaml_prompt

# ============================================================
# ONE-IMAGE OLMOCR-2 TEST
# ============================================================

image_dir = "/kaggle/input/datasets/manyabhatnagar/inputs1/ClinOCR_images"

# Get first image from dataframe
filename = os.path.basename(str(df.iloc[0]["image"]))
image_path = os.path.join(image_dir, filename)

print("Image:", image_path)
print("Exists:", os.path.exists(image_path))

# ------------------------------------------------------------
# Load image
# ------------------------------------------------------------

image = Image.open(image_path).convert("RGB")

print("Original size:", image.size)

# ------------------------------------------------------------
# Resize to olmOCR recommended 1288 longest dimension
# ------------------------------------------------------------

target_longest_dim = 1288

w, h = image.size

if max(w, h) > target_longest_dim:

    scale = target_longest_dim / max(w, h)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    image = image.resize(
        (new_w, new_h),
        Image.Resampling.LANCZOS
    )

print("Resized size :", image.size)

# ------------------------------------------------------------
# Build olmOCR prompt
# ------------------------------------------------------------

prompt = build_no_anchoring_v4_yaml_prompt()

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            },
            {
                "type": "image",
                "image": image
            }
        ]
    }
]

# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

print("\nINPUTS")

for key, value in inputs.items():
    if hasattr(value, "shape"):
        print(
            f"{key:20s} "
            f"{tuple(value.shape)} "
            f"{value.dtype} "
            f"{value.device}"
        )

# ------------------------------------------------------------
# Image grid / visual tokens
# ------------------------------------------------------------

if "image_grid_thw" in inputs:

    T, H, W = inputs["image_grid_thw"][0].tolist()

    print("\nIMAGE GRID")
    print("T =", T)
    print("H =", H)
    print("W =", W)
    print("Visual tokens =", int(T * H * W))

# ------------------------------------------------------------
# Put inputs on the model's INPUT device
# ------------------------------------------------------------

input_device = model.get_input_embeddings().weight.device

print("\nModel input device:", input_device)

inputs = {
    k: v.to(input_device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING OLMOCR-2...")
print("=" * 70)

with torch.inference_mode():

    output = model.generate(
        **inputs,
        temperature=0.1,
        max_new_tokens=4096,
        num_return_sequences=1,
        do_sample=True,
    )

# ------------------------------------------------------------
# Remove prompt tokens
# ------------------------------------------------------------

prompt_length = inputs["input_ids"].shape[1]

new_tokens = output[:, prompt_length:]

# ------------------------------------------------------------
# Decode
# ------------------------------------------------------------

result = processor.tokenizer.batch_decode(
    new_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)

prediction = result[0].strip()

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(prediction)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

In [1]:
def benchmark_olmocr2(
    df,
    model,
    processor,
    raw_output_dir="/kaggle/working/olmocr2_raw",
    image_dir="/kaggle/input/datasets/manyabhatnagar/inputs1/ClinOCR_images",
    max_new_tokens=2048,
    target_longest_dim=768,
    results_csv="/kaggle/working/olmocr2_results.csv",
):
    """
    Benchmark olmOCR-2.

    results_df = original df +:
        prediction
        runtime
        raw_output
        model

    Also saves:
        results_csv
        raw .txt outputs
        ZIP of raw outputs
    """

    import os
    import re
    import time
    import zipfile
    import warnings

    import torch
    from PIL import Image
    from tqdm.auto import tqdm

    from olmocr.prompts import build_no_anchoring_v4_yaml_prompt

    warnings.filterwarnings("ignore")

    # ============================================================
    # SETUP
    # ============================================================

    os.makedirs(raw_output_dir, exist_ok=True)

    # IMPORTANT:
    # Start with the ORIGINAL dataframe.
    results_df = df.copy()

    # Add ONLY the four benchmark columns
    for col in [
        "prediction",
        "runtime",
        "raw_output",
        "model",
    ]:
        if col not in results_df.columns:
            results_df[col] = None

    total = len(results_df)

    # ============================================================
    # RESUME
    # ============================================================

    completed = (
        results_df["prediction"].notna()
        &
        results_df["prediction"].astype(str).str.strip().ne("")
    )

    already_done = int(completed.sum())

    print("=" * 70)
    print("OLMOCR-2 BENCHMARK")
    print("=" * 70)
    print(f"Total documents : {total}")
    print(f"Already done    : {already_done}")
    print(f"Remaining       : {total - already_done}")
    print(f"Raw output dir  : {raw_output_dir}")
    print(f"Results CSV     : {results_csv}")
    print("=" * 70)

    # ============================================================
    # PROMPT
    # ============================================================

    prompt = build_no_anchoring_v4_yaml_prompt()

    # ============================================================
    # INPUT DEVICE
    # ============================================================

    input_device = model.get_input_embeddings().weight.device

    print(f"Model input device: {input_device}")

    # ============================================================
    # CUDA SYNC
    # ============================================================

    def sync_cuda():

        if torch.cuda.is_available():

            for gpu_id in range(torch.cuda.device_count()):

                try:
                    torch.cuda.synchronize(gpu_id)
                except Exception:
                    pass

    # ============================================================
    # TQDM
    # ============================================================

    pbar = tqdm(
        total=total,
        initial=already_done,
        desc="OlmOCR-2",
        unit="doc",
        dynamic_ncols=True,
        leave=True,
    )

    successful = already_done
    failed = 0

    # ============================================================
    # BENCHMARK
    # ============================================================

    for counter, idx in enumerate(results_df.index, start=1):

        if completed.loc[idx]:
            continue

        row = results_df.loc[idx]

        doc_id = str(row["doc_id"])

        filename = os.path.basename(
            str(row["image"])
        )

        image_path = os.path.join(
            image_dir,
            filename
        )

        pbar.set_postfix_str(
            f"current={doc_id} | loading"
        )

        # --------------------------------------------------------
        # IMAGE CHECK
        # --------------------------------------------------------

        if not os.path.exists(image_path):

            failed += 1

            results_df.at[idx, "prediction"] = ""
            results_df.at[idx, "runtime"] = None
            results_df.at[idx, "raw_output"] = None
            results_df.at[idx, "model"] = "olmOCR-2"

            pbar.update(1)

            pbar.set_postfix_str(
                f"done={successful} | "
                f"failed={failed} | "
                f"{doc_id} | IMAGE NOT FOUND"
            )

            continue

        raw_filename = f"{doc_id}.txt"

        raw_path = os.path.join(
            raw_output_dir,
            raw_filename
        )

        try:

            # ====================================================
            # LOAD IMAGE
            # ====================================================

            image = Image.open(
                image_path
            ).convert("RGB")

            # ====================================================
            # RESIZE
            # ====================================================

            if max(image.size) > target_longest_dim:

                scale = (
                    target_longest_dim
                    / max(image.size)
                )

                new_w = int(
                    round(image.width * scale)
                )

                new_h = int(
                    round(image.height * scale)
                )

                image = image.resize(
                    (new_w, new_h),
                    Image.Resampling.LANCZOS
                )

            # ====================================================
            # MESSAGE
            # ====================================================

            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt,
                        },
                        {
                            "type": "image",
                            "image": image,
                        },
                    ],
                }
            ]

            # ====================================================
            # PROCESS
            # ====================================================

            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )

            # ====================================================
            # MOVE INPUTS
            # ====================================================

            inputs = {
                k: v.to(input_device)
                if hasattr(v, "to")
                else v
                for k, v in inputs.items()
            }

            # ====================================================
            # TIMER
            # ====================================================

            sync_cuda()

            start = time.perf_counter()

            # ====================================================
            # INFERENCE
            # ====================================================

            with torch.inference_mode():

                output = model.generate(
                    **inputs,
                    temperature=0.1,
                    max_new_tokens=max_new_tokens,
                    num_return_sequences=1,
                    do_sample=True,
                )

            sync_cuda()

            runtime = time.perf_counter() - start

            # ====================================================
            # REMOVE PROMPT
            # ====================================================

            prompt_length = inputs["input_ids"].shape[1]

            new_tokens = output[:, prompt_length:]

            # ====================================================
            # DECODE
            # ====================================================

            decoded = processor.tokenizer.batch_decode(
                new_tokens,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if not decoded:
                raise RuntimeError(
                    "olmOCR-2 returned an empty result."
                )

            raw_text = decoded[0].strip()

            # ====================================================
            # REMOVE YAML METADATA
            # ====================================================

            prediction = re.sub(
                r"^\s*---\s*\n.*?\n---\s*\n",
                "",
                raw_text,
                count=1,
                flags=re.DOTALL,
            ).strip()

            # ====================================================
            # SAVE RAW OUTPUT
            # ====================================================

            with open(
                raw_path,
                "w",
                encoding="utf-8",
            ) as f:
                f.write(raw_text)

            # ====================================================
            # STORE IN RESULTS DF
            # ====================================================

            results_df.at[
                idx, "prediction"
            ] = prediction

            results_df.at[
                idx, "runtime"
            ] = runtime

            results_df.at[
                idx, "raw_output"
            ] = raw_filename

            results_df.at[
                idx, "model"
            ] = "olmOCR-2"

            successful += 1

            # ====================================================
            # PROGRESS
            # ====================================================

            pbar.update(1)

            pbar.set_postfix_str(
                f"done={successful} | "
                f"failed={failed} | "
                f"{doc_id} | "
                f"{runtime:.2f}s"
            )

        # ========================================================
        # ERROR
        # ========================================================

        except Exception as e:

            failed += 1

            results_df.at[
                idx, "prediction"
            ] = ""

            results_df.at[
                idx, "runtime"
            ] = None

            results_df.at[
                idx, "raw_output"
            ] = None

            results_df.at[
                idx, "model"
            ] = "olmOCR-2"

            pbar.update(1)

            pbar.set_postfix_str(
                f"done={successful} | "
                f"failed={failed} | "
                f"{doc_id} | "
                f"ERROR: {type(e).__name__}"
            )

    pbar.close()

    # ============================================================
    # FINAL COUNTS
    # ============================================================

    successful_mask = (
        results_df["prediction"].notna()
        &
        results_df["prediction"].astype(str).str.strip().ne("")
    )

    success_count = int(successful_mask.sum())
    failed_count = total - success_count

    # ============================================================
    # SAVE COMPLETE RESULTS DATAFRAME
    # ============================================================

    results_df.to_csv(
        results_csv,
        index=False,
        encoding="utf-8",
    )

    # ============================================================
    # ZIP RAW OUTPUTS
    # ============================================================

    zip_path = "/kaggle/working/olmocr2_raw.zip"

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as zipf:

        for filename in os.listdir(raw_output_dir):

            filepath = os.path.join(
                raw_output_dir,
                filename
            )

            if os.path.isfile(filepath):

                zipf.write(
                    filepath,
                    arcname=filename
                )

    # ============================================================
    # FINAL
    # ============================================================

    print()
    print("=" * 70)
    print("OLMOCR-2 BENCHMARK COMPLETE")
    print("=" * 70)
    print(f"Total documents : {total}")
    print(f"Successful      : {success_count}")
    print(f"Failed          : {failed_count}")
    print(f"Rows in CSV     : {len(results_df)}")
    print(f"Columns in CSV  : {len(results_df.columns)}")
    print(f"CSV             : {results_csv}")
    print(f"Raw outputs     : {raw_output_dir}")
    print(f"ZIP             : {zip_path}")
    print("=" * 70)

    return results_df

In [ ]:
olmocr2_results = benchmark_olmocr2(
    df=df,
    model=model,
    processor=processor,
    max_new_tokens=2048,
    target_longest_dim=1024,
)